In [516]:
import os

import importlib
import json
import pandas as pd
import random
import time

from datetime import date
from dotenv import load_dotenv
from openai import OpenAI

import parsing
import evaluation
import exporting

from parsing import parse_output
from evaluation import evaluation_v1, evaluation_v2
from exporting import export_run

In [517]:
random.seed(42)

In [518]:
importlib.reload(parsing)
from parsing import parse_output

importlib.reload(evaluation)
from evaluation import evaluation_v1

importlib.reload(exporting)
from exporting import export_run

: 

In [ ]:
EVALUATORS = {
    "v1": evaluation_v1,
    "v2": evaluation_v2,
    "v3": evaluation_v3
}

# BioRED Train & GT Loading & Parsing

### Load Key

In [299]:
load_dotenv()
print(os.getenv("OPEN_AI_TEST_KEY")[:15])

sk-proj-4WDSBIA


In [3]:
client = OpenAI(
    api_key=os.getenv("OPEN_AI_TEST_KEY")
)

### Load & Downsample BioRED Train Abstracts

In [4]:
biored_train = pd.read_csv("../data/processed/biored/br_train.csv")

biored_train_sample = biored_train.sample(
    n=35,
    random_state=42
).copy()

### BioRED GT Parsing

In [145]:
sampled_pmids = set(biored_train_sample["pmid"])

biored_train_gts = pd.read_csv("../data/processed/biored/br_train_entity_relations.csv")

biored_train_gts_filtered = biored_train_gts[biored_train_gts["pmid"].isin(sampled_pmids)]

### Parse Into Two Sets of Tuples
* `predictions_entities`
* `predictions_relationships`

**Includes PMID for duplicate handling as sets automatically remove duplicates**
* PMID avoids duplicates across papers

In [147]:
ground_truth_entities = (
    set(zip(biored_train_gts_filtered["pmid"], biored_train_gts_filtered["entity_1"].str.strip().str.lower(), biored_train_gts_filtered["entity_1_type"]))
    | set(zip(biored_train_gts_filtered["pmid"], biored_train_gts_filtered["entity_2"].str.strip().str.lower(), biored_train_gts_filtered["entity_2_type"]))
)

ground_truth_relationships = set(zip(
    biored_train_gts_filtered["pmid"],
    biored_train_gts_filtered["entity_1"].str.strip().str.lower(),
    biored_train_gts_filtered["relation"],
    biored_train_gts_filtered["entity_2"].str.strip().str.lower()
))

In [238]:
ground_truths = {
    "entities": ground_truth_entities,
    "relationships": ground_truth_relationships
}

### Few-Shot Construction

In [179]:
# Exclude the 35 pmids you're evaluating on
non_eval_pmids = set(biored_train["pmid"]) - sampled_pmids
few_shot_pmids = random.sample(list(non_eval_pmids), 3)

def build_few_shot_example(pmid, example_num):
    abstract = biored_train[biored_train["pmid"] == pmid]["abstract"].iloc[0]
    gt_rows = biored_train_gts[biored_train_gts["pmid"] == pmid]

    entities = list({
        (row["entity_1"], row["entity_1_type"]) for _, row in gt_rows.iterrows()
    } | {
        (row["entity_2"], row["entity_2_type"]) for _, row in gt_rows.iterrows()
    })

    relationships = [
        {"source": row["entity_1"], "relation": row["relation"], "target": row["entity_2"]}
        for _, row in gt_rows.iterrows()
    ]

    output_json = {
        "entities": [{"text": e[0], "type": e[1]} for e in entities],
        "relationships": relationships
    }

    return f"## EXAMPLE {example_num+1}:\n\n### Abstract:\n\n{abstract}\n\n### Correct BioRED annotation:\n\n{json.dumps(output_json, indent=2)}"

few_shot_block = "\n\n".join(build_few_shot_example(pmid, i) for i, pmid in enumerate(few_shot_pmids))

print(few_shot_block[:1750])  # sanity check

## EXAMPLE 1:

### Abstract:

To determine the incidence of clinically significant adverse events after long-term, fixed-dose, generic highly active antiretroviral therapy (HAART) use among HIV-infected individuals in South India, we examined the experiences of 3154 HIV-infected individuals who received a minimum of 3 months of generic HAART between February 1996 and December 2006 at a tertiary HIV care referral center in South India. The most common regimens were 3TC + d4T + nevirapine (NVP) (54.8%), zidovudine (AZT) + 3TC + NVP (14.5%), 3TC + d4T + efavirenz (EFV) (20.1%), and AZT + 3TC + EFV (5.4%). The most common adverse events and median CD4 at time of event were rash (15.2%; CD4, 285 cells/microL) and peripheral neuropathy (9.0% and 348 cells/microL). Clinically significant anemia (hemoglobin <7 g/dL) was observed in 5.4% of patients (CD4, 165 cells/microL) and hepatitis (clinical jaundice with alanine aminotransferase > 5 times upper limits of normal) in 3.5% of patients (CD4, 

# LLM API Call

In [450]:
with open("../data/prompt_refinement/prompt_versions.json", "r") as f:
    PROMPTS = json.load(f)

In [493]:
PROMPT_VERSION = "v3"
EVAL_VERSION = "v2"
RUN_NOTES = "Implemented BioRED entity and relation schema and rules on top of prompt v1."

In [ ]:
start_time = time.perf_counter()

outputs = []

for index, row in biored_train_sample.iterrows():
    abstract = row["abstract"]

    prompt = (
        PROMPTS[PROMPT_VERSION]["template"]
        .replace("{abstract}", abstract)
        .replace("{few_shot_block}", few_shot_block)
    )

    response = client.responses.create(
        model="gpt-5.6-luna",
        input=prompt
    )

    outputs.append({
        "pmid": row["pmid"],
        "output": response.output_text
    })

In [ ]:
elapsed_seconds = time.perf_counter() - start_time
print(f"API calls took {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f} min) for {len(outputs)} abstracts")

In [495]:
output_dict = {
    "outputs": outputs,
    "run_notes": RUN_NOTES,
    "prompt_version": PROMPT_VERSION
}

### Parse `outputs` list to JSON

In [496]:
run_dict = parse_output(output_dict)

Parsed 35 extractions, 0 failed to parse as JSON


# Evaluation Metrics

In [497]:
evaluator = EVALUATORS[EVAL_VERSION]
eval_results = evaluator(run_dict, ground_truths)

# Export Run

### Create Run Storage Dir.

In [498]:
os.makedirs("../data/exploration", exist_ok=True)

In [499]:
raw_prompt = PROMPTS[PROMPT_VERSION]["template"]

In [ ]:
export_run(run_dict, eval_results, prompt=raw_prompt, time_taken=elapsed_seconds);

Saved run_004 to ../prompt_runs


# Error Sampling

In [501]:
def sample_errors(predictions, ground_truth, n=20, label="items"):
    false_positives = list(predictions - ground_truth)
    false_negatives = list(ground_truth - predictions)

    fp_sample = random.sample(false_positives, min(n, len(false_positives)))
    fn_sample = random.sample(false_negatives, min(n, len(false_negatives)))

    print(f"--- {label}: False Positives (predicted, not in ground truth) ---")
    print(f"Sampled {len(fp_sample)} of {len(false_positives)} total FPs\n")
    for item in fp_sample:
        print(" ", item)

    print(f"\n--- {label}: False Negatives (in ground truth, not predicted) ---")
    print(f"Sampled {len(fn_sample)} of {len(false_negatives)} total FNs\n")
    for item in fn_sample:
        print(" ", item)

    return fp_sample, fn_sample

In [502]:
relation_fp_sample, relation_fn_sample = sample_errors(
    run_dict["predictions_relationships"],
    ground_truth_relationships,
    n=20,
    label="Relationships"
)

--- Relationships: False Positives (predicted, not in ground truth) ---
Sampled 20 of 333 total FPs

  (18808529, 'dystrophin-glycoprotein complex (dgc)', 'Bind', 'laminin')
  (18808529, 'lysis of myofilaments', 'Negative_Correlation', 'laminin alpha-2')
  (15970799, '*15+c1007g', 'Negative_Correlation', 'estradiol-17beta-d-glucuronide')
  (20510337, 'coenzyme q10', 'Positive_Correlation', 'superoxide dismutase')
  (19918264, 'fibroblast growth factor receptor 4', 'Association', 'prostate cancer')
  (15970799, 'slco1b1*15', 'Negative_Correlation', 'cerivastatin')
  (20510337, 'cisplatin', 'Negative_Correlation', 'selenium')
  (28512644, 'cat c262t', 'Association', 'ccl2')
  (16120104, '111g', 'Association', 'advanced sleep phase syndrome')
  (24743235, 'csf-1', 'Positive_Correlation', 'cd11c+ cell')
  (24309294, 'l-364,718', 'Bind', 'cck1 receptor')
  (18827003, 'd401h', 'Association', 'hypertension')
  (17975693, 'ciprofloxacin', 'Comparison', 'norfloxacin')
  (24914936, 't4', 'Associ

In [503]:
entity_fp_sample, entity_fn_sample = sample_errors(
    run_dict["predictions_entities"],
    ground_truth_entities,
    n=20,
    label="Entities"
)

--- Entities: False Positives (predicted, not in ground truth) ---
Sampled 20 of 278 total FPs

  (24743235, 'il-6', 'GeneOrGeneProduct')
  (25305591, 'vasoactive intestinal peptide', 'ChemicalEntity')
  (16120104, 'glycine', 'ChemicalEntity')
  (15970799, '*1b+c1007g', 'SequenceVariant')
  (28428256, 'zo-1', 'GeneOrGeneProduct')
  (17192049, 'ile462val', 'SequenceVariant')
  (24914936, 'bone stiffness', 'DiseaseOrPhenotypicFeature')
  (18827003, 'tissue-selective glucocorticoid hypersensitivity', 'DiseaseOrPhenotypicFeature')
  (10491763, 'insulin responses to glucose', 'DiseaseOrPhenotypicFeature')
  (19108278, '(-)-propranolol', 'ChemicalEntity')
  (28411266, 'bmi', 'DiseaseOrPhenotypicFeature')
  (25305591, 'vip', 'ChemicalEntity')
  (20431083, 'antithrombotic drugs', 'ChemicalEntity')
  (15970799, 'slco1b1*1b', 'SequenceVariant')
  (19108278, 'pr interval', 'DiseaseOrPhenotypicFeature')
  (21163864, 'left ventricular hypertrophy', 'DiseaseOrPhenotypicFeature')
  (15099351, 'low-de